<a href="https://colab.research.google.com/github/Saisneha0209/Own-Practise/blob/main/Self_HI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from dataclasses import dataclass
from typing import List, Dict
import random
import statistics

@dataclass
class Event:
    event_id: int
    source: str
    value: float
    previous_value: float
    context: str

    important: bool = False

    @property
    def novelty(self):
        return min(abs(self.value - self.previous_value), 1.0)

    @property
    def magnitude(self):
        return min(abs(self.value), 1.0)


class FixedFilter:

    def __init__(self, processing_budget=20):
        self.processing_budget = processing_budget

    def score(self, event):

        return (
            0.50 * event.novelty +
            0.50 * event.magnitude
        )

    def select(self, events):

        ranked = sorted(
            events,
            key=self.score,
            reverse=True
        )

        return ranked[:self.processing_budget]



class HumanoidFilter:

    def __init__(
        self,
        processing_budget=20,
        recheck_fraction=0.30
    ):

        self.processing_budget = processing_budget
        self.recheck_fraction = recheck_fraction

        self.active_context = "navigation"

        # Explicit symbolic priorities.
        # These are NOT ML weights.
        self.source_priority = {
            "vision": 0.90,
            "audio": 0.70,
            "motion": 1.00,
            "temperature": 0.50
        }

    # --------------------------------------------------------
    # INITIAL ATTENTION
    # --------------------------------------------------------

    def first_pass_score(self, event):

        novelty = event.novelty
        magnitude = event.magnitude

        priority = self.source_priority.get(
            event.source,
            0.50
        )

        if event.context == self.active_context:
            context_relevance = 1.0
        else:
            context_relevance = 0.25

        score = (
            0.40 * novelty +
            0.30 * magnitude +
            0.20 * priority +
            0.10 * context_relevance
        )

        return score

    # --------------------------------------------------------
    # DELIBERATE RECHECKING
    # --------------------------------------------------------

    def recheck_score(self, event, events):

        original_score = self.first_pass_score(event)

        # Look at nearby events.
        left = max(0, event.event_id - 5)
        right = min(
            len(events),
            event.event_id + 6
        )

        nearby = events[left:right]

        same_source_events = [
            other
            for other in nearby
            if (
                other.source == event.source
                and other.event_id != event.event_id
            )
        ]

        # Is there supporting evidence nearby?
        if same_source_events:

            corroboration = max(
                x.novelty
                for x in same_source_events
            )

        else:

            corroboration = 0.0

        # Context can strengthen attention.
        context_boost = 0

        if event.context == self.active_context:
            context_boost = 0.15

        score = (
            original_score
            + 0.18 * corroboration
            + context_boost
        )

        return min(score, 1.0)

    # --------------------------------------------------------
    # EVENT SELECTION
    # --------------------------------------------------------

    def select(self, events):

        # Reserve some processing capacity for rechecking.

        first_budget = round(
            self.processing_budget
            * (1 - self.recheck_fraction)
        )

        recheck_budget = (
            self.processing_budget
            - first_budget
        )

        # -----------------------------
        # Fast filtering
        # -----------------------------

        ranked = sorted(
            events,
            key=self.first_pass_score,
            reverse=True
        )

        selected = ranked[:first_budget]

        # -----------------------------
        # Deferred attention
        # -----------------------------

        remaining = ranked[
            first_budget:
            first_budget + self.processing_budget * 3
        ]

        # -----------------------------
        # Deliberate recheck
        # -----------------------------

        reconsidered = sorted(
            remaining,
            key=lambda e:
                self.recheck_score(e, events),
            reverse=True
        )

        selected.extend(
            reconsidered[:recheck_budget]
        )

        # Never exceed processing budget.

        return selected[
            :self.processing_budget
        ]


# ============================================================
# PROCEDURAL MEMORY
# ============================================================

class RuleMemory:

    def __init__(self):

        self.rules = []

    def add_rule(
        self,
        name,
        condition,
        action
    ):

        self.rules.append(
            (
                name,
                condition,
                action
            )
        )

    def execute(self, event):

        responses = []

        for name, condition, action in self.rules:

            if condition(event):

                responses.append(
                    {
                        "rule": name,
                        "action": action(event)
                    }
                )

        return responses


# ============================================================
# BUILD BASIC "LEARNED" PROCEDURES
# ============================================================

def build_memory():

    memory = RuleMemory()

    # Rule 1

    memory.add_rule(

        "ObstacleRule",

        lambda event:
            event.source == "motion"
            and event.value > 0.72,

        lambda event:
            "Possible obstacle detected"
    )

    # Rule 2

    memory.add_rule(

        "VisualChangeRule",

        lambda event:
            event.source == "vision"
            and
            event.context == "navigation"
            and
            event.novelty > 0.43,

        lambda event:
            "Unexpected visual change"
    )

    # Rule 3

    memory.add_rule(

        "AudioAlertRule",

        lambda event:
            event.source == "audio"
            and event.value > 0.88,

        lambda event:
            "Strong audio event"
    )

    return memory


# ============================================================
# SIMULATED ENVIRONMENT
# ============================================================

def generate_events(
    number_of_events=120,
    seed=7
):

    random.seed(seed)

    sources = [
        "vision",
        "audio",
        "motion",
        "temperature"
    ]

    contexts = [
        "navigation",
        "conversation",
        "idle"
    ]

    previous_values = {
        source: 0.10
        for source in sources
    }

    events = []

    for event_id in range(
        number_of_events
    ):

        source = random.choice(sources)

        context = random.choice(contexts)

        value = random.random()

        previous = previous_values[source]

        previous_values[source] = value

        novelty = abs(
            value - previous
        )

        # ====================================================
        # Hidden experimental ground truth
        #
        # The filter DOES NOT receive this information.
        # It is only used to evaluate whether the system
        # successfully detected relevant information.
        # ====================================================

        important = (

            (
                source == "motion"
                and value > 0.72
            )

            or

            (
                source == "vision"
                and context == "navigation"
                and novelty > 0.43
            )

            or

            (
                source == "audio"
                and context == "conversation"
                and value > 0.88
            )

            or

            (
                source == "temperature"
                and value > 0.96
            )
        )

        events.append(

            Event(

                event_id=event_id,

                source=source,

                value=value,

                previous_value=previous,

                context=context,

                important=important
            )
        )

    return events


# ============================================================
# EVALUATION
# ============================================================

def evaluate(
    selected,
    all_events
):

    important_ids = {

        event.event_id

        for event in all_events

        if event.important
    }

    selected_ids = {

        event.event_id

        for event in selected
    }

    true_positive = len(
        important_ids
        & selected_ids
    )

    false_positive = len(
        selected_ids
        - important_ids
    )

    false_negative = len(
        important_ids
        - selected_ids
    )

    if (
        true_positive
        + false_negative
    ):

        recall = (

            true_positive
            /
            (
                true_positive
                + false_negative
            )
        )

    else:

        recall = 0

    if (
        true_positive
        + false_positive
    ):

        precision = (

            true_positive
            /
            (
                true_positive
                + false_positive
            )
        )

    else:

        precision = 0

    return {

        "processed":
            len(selected_ids),

        "important_detected":
            true_positive,

        "recall":
            recall,

        "precision":
            precision
    }


# ============================================================
# BENCHMARK
# ============================================================

def benchmark(
    trials=200,
    budget=20
):

    fixed_recalls = []

    hi_recalls = []

    fixed_precisions = []

    hi_precisions = []

    for seed in range(trials):

        events = generate_events(
            seed=seed
        )

        # -------------------------------
        # Fixed architecture
        # -------------------------------

        fixed = FixedFilter(
            processing_budget=budget
        )

        fixed_selected = (
            fixed.select(events)
        )

        fixed_result = evaluate(
            fixed_selected,
            events
        )

        # -------------------------------
        # Humanoid Intelligence
        # -------------------------------

        humanoid = HumanoidFilter(
            processing_budget=budget,
            recheck_fraction=0.30
        )

        hi_selected = (
            humanoid.select(events)
        )

        hi_result = evaluate(
            hi_selected,
            events
        )

        fixed_recalls.append(
            fixed_result["recall"]
        )

        hi_recalls.append(
            hi_result["recall"]
        )

        fixed_precisions.append(
            fixed_result["precision"]
        )

        hi_precisions.append(
            hi_result["precision"]
        )

    print(
        "\n======== EXPERIMENT ========"
    )

    print(
        "Trials:",
        trials
    )

    print(
        "Processing budget:",
        budget
    )

    print()

    print(
        "FIXED FILTER"
    )

    print(
        "Average recall:",
        round(
            statistics.mean(
                fixed_recalls
            ),
            3
        )
    )

    print(
        "Average precision:",
        round(
            statistics.mean(
                fixed_precisions
            ),
            3
        )
    )

    print()

    print(
        "HUMANOID INTELLIGENCE"
    )

    print(
        "Average recall:",
        round(
            statistics.mean(
                hi_recalls
            ),
            3
        )
    )

    print(
        "Average precision:",
        round(
            statistics.mean(
                hi_precisions
            ),
            3
        )
    )


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    benchmark(
        trials=200,
        budget=20
    )


======== EXPERIMENT ========
Trials: 200
Processing budget: 20

FIXED FILTER
Average recall: 0.548
Average precision: 0.373

HUMANOID INTELLIGENCE
Average recall: 0.589
Average precision: 0.4


#2nd simulation test


In [7]:
from dataclasses import dataclass
import random
import statistics
from typing import Callable, List, Dict, Tuple


# ============================================================
# HUMANOID INTELLIGENCE - HI-0
#
# Enhanced experimental prototype
#
# Main ideas:
#
# 1. Event-driven representation
# 2. Fixed filtering baseline
# 3. Context-aware attention
# 4. Same-source deliberate rechecking
# 5. Cross-modal deliberate rechecking
# 6. Equal processing budgets
# 7. Procedural rule memory
# 8. Experimental evaluation
#
# NOTE:
# This is a research simulation.
# It does NOT prove general intelligence.
# ============================================================


# ============================================================
# 1. EVENT REPRESENTATION
# ============================================================

@dataclass
class Event:

    event_id: int

    source: str

    value: float

    previous_value: float

    context: str

    # Experimental answer key only.
    # HI is NOT allowed to directly use this field.
    important: bool = False


    # --------------------------------------------------------
    # NOVELTY
    #
    # How much did the signal change?
    # --------------------------------------------------------

    @property
    def novelty(self) -> float:

        difference = abs(
            self.value
            -
            self.previous_value
        )

        return min(
            difference,
            1.0
        )


    # --------------------------------------------------------
    # MAGNITUDE
    #
    # How strong is the current signal?
    # --------------------------------------------------------

    @property
    def magnitude(self) -> float:

        return min(
            abs(self.value),
            1.0
        )


# ============================================================
# 2. BASELINE
#
# FIXED FILTER
#
# This represents a simple system whose filtering rule
# does NOT change according to context.
# ============================================================

class FixedFilter:

    def __init__(
        self,
        processing_budget: int = 20
    ):

        self.processing_budget = (
            processing_budget
        )


    # --------------------------------------------------------
    # FIXED ATTENTION SCORE
    # --------------------------------------------------------

    def score(
        self,
        event: Event
    ) -> float:

        score = (

            0.50
            * event.novelty

            +

            0.50
            * event.magnitude
        )

        return score


    # --------------------------------------------------------
    # SELECT TOP EVENTS
    # --------------------------------------------------------

    def select(
        self,
        events: List[Event]
    ) -> List[Event]:

        ranked = sorted(

            events,

            key=self.score,

            reverse=True
        )

        return ranked[
            :self.processing_budget
        ]


# ============================================================
# 3. HUMANOID INTELLIGENCE FILTER
# ============================================================

class HumanoidFilter:

    """
    Experimental HI attention architecture.

    Modes:

    context_only
        Contextual filtering without deliberate rechecking.

    same_source
        Recheck using nearby events from the same sensor.

    cross_modal
        Enhanced rechecking using relationships between
        different sensor modalities.
    """


    def __init__(
        self,
        processing_budget: int = 20,
        recheck_fraction: float = 0.30,
        mode: str = "cross_modal",
        active_context: str = "navigation"
    ):


        # ----------------------------------------------------
        # VALIDATION
        # ----------------------------------------------------

        if processing_budget <= 0:

            raise ValueError(
                "processing_budget must be greater than 0"
            )


        if not 0.0 <= recheck_fraction <= 1.0:

            raise ValueError(
                "recheck_fraction must be between 0 and 1"
            )


        valid_modes = {

            "context_only",

            "same_source",

            "cross_modal"
        }


        if mode not in valid_modes:

            raise ValueError(
                f"mode must be one of {valid_modes}"
            )


        # ----------------------------------------------------
        # SYSTEM STATE
        # ----------------------------------------------------

        self.processing_budget = (
            processing_budget
        )

        self.mode = mode

        self.active_context = (
            active_context
        )


        # Context-only mode performs no second pass.

        if mode == "context_only":

            self.recheck_fraction = 0.0

        else:

            self.recheck_fraction = (
                recheck_fraction
            )


        # ----------------------------------------------------
        # SOURCE PRIORITY
        #
        # These are manually defined prototype parameters.
        #
        # They are NOT learned neural-network weights.
        # ----------------------------------------------------

        self.source_priority = {

            "vision":
                0.90,

            "audio":
                0.70,

            "motion":
                1.00,

            "temperature":
                0.50
        }


        # ----------------------------------------------------
        # CROSS-MODAL RELATIONSHIPS
        #
        # Example:
        #
        # vision + motion
        #
        # may provide stronger evidence than
        #
        # vision alone.
        #
        # Again, these values are experimental parameters.
        # ----------------------------------------------------

        self.cross_modal_links = {

            "vision": {

                "motion": 1.00,

                "audio": 0.60
            },


            "motion": {

                "vision": 1.00,

                "audio": 0.75
            },


            "audio": {

                "motion": 0.75,

                "vision": 0.60
            },


            "temperature": {}
        }


    # ========================================================
    # 4. FIRST-PASS ATTENTION
    # ========================================================

    def first_pass_score(
        self,
        event: Event
    ) -> float:


        novelty = (
            event.novelty
        )


        magnitude = (
            event.magnitude
        )


        priority = (
            self.source_priority.get(
                event.source,
                0.50
            )
        )


        # ----------------------------------------------------
        # CONTEXT RELEVANCE
        # ----------------------------------------------------

        if (
            event.context
            ==
            self.active_context
        ):

            context_relevance = 1.0

        else:

            context_relevance = 0.25


        # ----------------------------------------------------
        # ATTENTION EQUATION
        # ----------------------------------------------------

        score = (

            0.40
            * novelty

            +

            0.30
            * magnitude

            +

            0.20
            * priority

            +

            0.10
            * context_relevance
        )


        return min(
            score,
            1.0
        )


    # ========================================================
    # 5. FIND TEMPORALLY NEARBY EVENTS
    # ========================================================

    def nearby_events(
        self,
        event: Event,
        events: List[Event],
        radius: int = 5
    ) -> List[Event]:


        left = max(

            0,

            event.event_id
            -
            radius
        )


        right = min(

            len(events),

            event.event_id
            +
            radius
            +
            1
        )


        nearby = [

            other

            for other
            in events[left:right]

            if (
                other.event_id
                !=
                event.event_id
            )
        ]


        return nearby


    # ========================================================
    # 6. SAME-SOURCE RECHECKING
    #
    # Original rechecking idea.
    # ========================================================

    def same_source_recheck_score(
        self,
        event: Event,
        events: List[Event]
    ) -> float:


        original_score = (
            self.first_pass_score(
                event
            )
        )


        nearby = (
            self.nearby_events(
                event,
                events
            )
        )


        same_source = [

            other

            for other in nearby

            if (
                other.source
                ==
                event.source
            )
        ]


        # ----------------------------------------------------
        # Look for strongest nearby signal
        # from same modality.
        # ----------------------------------------------------

        if same_source:

            corroboration = max(

                other.novelty

                for other
                in same_source
            )

        else:

            corroboration = 0.0


        # ----------------------------------------------------
        # CONTEXT BOOST
        # ----------------------------------------------------

        if (
            event.context
            ==
            self.active_context
        ):

            context_boost = 0.15

        else:

            context_boost = 0.0


        score = (

            original_score

            +

            0.18
            * corroboration

            +

            context_boost
        )


        return min(
            score,
            1.0
        )


    # ========================================================
    # 7. ENHANCED CROSS-MODAL RECHECKING
    # ========================================================

    def cross_modal_recheck_score(
        self,
        event: Event,
        events: List[Event]
    ) -> float:


        original_score = (
            self.first_pass_score(
                event
            )
        )


        nearby = (
            self.nearby_events(
                event,
                events
            )
        )


        # ----------------------------------------------------
        # Find which other modalities can support
        # the current modality.
        # ----------------------------------------------------

        linked_sources = (
            self.cross_modal_links.get(
                event.source,
                {}
            )
        )


        cross_modal_evidence = 0.0


        # ----------------------------------------------------
        # SEARCH FOR SUPPORTING SIGNALS
        # ----------------------------------------------------

        for other in nearby:


            relationship_strength = (
                linked_sources.get(
                    other.source,
                    0.0
                )
            )


            if relationship_strength > 0:


                evidence = (

                    relationship_strength

                    *

                    (

                        0.60
                        * other.novelty

                        +

                        0.40
                        * other.magnitude
                    )
                )


                cross_modal_evidence = max(

                    cross_modal_evidence,

                    evidence
                )


        # ----------------------------------------------------
        # SAME-SOURCE SUPPORT
        #
        # Still useful, but deliberately weaker.
        # ----------------------------------------------------

        same_source_evidence = max(

            (

                other.novelty

                for other in nearby

                if (
                    other.source
                    ==
                    event.source
                )
            ),

            default=0.0
        )


        # ----------------------------------------------------
        # CONTEXT BOOST
        # ----------------------------------------------------

        if (
            event.context
            ==
            self.active_context
        ):

            context_boost = 0.12

        else:

            context_boost = 0.0


        # ----------------------------------------------------
        # MODALITY DIVERSITY
        #
        # Multiple different active sources may indicate
        # a meaningful environmental occurrence.
        # ----------------------------------------------------

        active_sources = {

            other.source

            for other in nearby

            if (
                other.novelty
                >
                0.35
            )
        }


        diversity_boost = min(

            0.04
            * len(active_sources),

            0.12
        )


        # ----------------------------------------------------
        # FINAL RECHECK SCORE
        # ----------------------------------------------------

        score = (

            original_score

            +

            0.22
            * cross_modal_evidence

            +

            0.08
            * same_source_evidence

            +

            context_boost

            +

            diversity_boost
        )


        return min(
            score,
            1.0
        )


    # ========================================================
    # 8. EVENT SELECTION
    #
    # IMPORTANT:
    #
    # Total processing budget remains equal.
    # ========================================================

    def select(
        self,
        events: List[Event]
    ) -> List[Event]:


        # ----------------------------------------------------
        # FIRST-PASS RANKING
        # ----------------------------------------------------

        ranked = sorted(

            events,

            key=self.first_pass_score,

            reverse=True
        )


        # ----------------------------------------------------
        # DIVIDE BUDGET
        # ----------------------------------------------------

        first_budget = round(

            self.processing_budget

            *

            (
                1
                -
                self.recheck_fraction
            )
        )


        recheck_budget = (

            self.processing_budget

            -

            first_budget
        )


        # ----------------------------------------------------
        # INITIAL SELECTION
        # ----------------------------------------------------

        selected = ranked[
            :first_budget
        ]


        # ----------------------------------------------------
        # CONTEXT-ONLY MODE
        # ----------------------------------------------------

        if recheck_budget == 0:

            return selected[
                :self.processing_budget
            ]


        # ----------------------------------------------------
        # DEFERRED CANDIDATES
        # ----------------------------------------------------

        candidate_pool_size = (

            self.processing_budget
            *
            3
        )


        remaining = ranked[

            first_budget

            :

            first_budget
            +
            candidate_pool_size
        ]


        # ----------------------------------------------------
        # CHOOSE RECHECKING ALGORITHM
        # ----------------------------------------------------

        if self.mode == "same_source":

            recheck_function = (
                self.same_source_recheck_score
            )

        else:

            recheck_function = (
                self.cross_modal_recheck_score
            )


        # ----------------------------------------------------
        # SECOND ATTENTION PASS
        # ----------------------------------------------------

        reconsidered = sorted(

            remaining,

            key=lambda event:
                recheck_function(
                    event,
                    events
                ),

            reverse=True
        )


        selected.extend(

            reconsidered[
                :recheck_budget
            ]
        )


        # ----------------------------------------------------
        # SAFETY:
        #
        # Never exceed processing budget.
        # ----------------------------------------------------

        return selected[
            :self.processing_budget
        ]


# ============================================================
# 9. PROCEDURAL MEMORY
# ============================================================

class RuleMemory:


    def __init__(self):


        self.rules: List[

            Tuple[

                str,

                Callable[
                    [Event],
                    bool
                ],

                Callable[
                    [Event],
                    str
                ]
            ]

        ] = []


    # --------------------------------------------------------
    # ADD PROCEDURAL RULE
    # --------------------------------------------------------

    def add_rule(
        self,
        name: str,
        condition: Callable[[Event], bool],
        action: Callable[[Event], str]
    ) -> None:


        self.rules.append(

            (

                name,

                condition,

                action
            )
        )


    # --------------------------------------------------------
    # EXECUTE RELEVANT RULES
    # --------------------------------------------------------

    def execute(
        self,
        event: Event
    ) -> List[Dict[str, str]]:


        responses = []


        for (
            name,
            condition,
            action
        ) in self.rules:


            if condition(event):


                responses.append(

                    {

                        "rule":
                            name,

                        "action":
                            action(event)
                    }
                )


        return responses


# ============================================================
# 10. BUILD PROCEDURAL KNOWLEDGE
# ============================================================

def build_memory() -> RuleMemory:


    memory = RuleMemory()


    # --------------------------------------------------------
    # RULE 1
    # --------------------------------------------------------

    memory.add_rule(

        "ObstacleRule",

        lambda event: (

            event.source
            ==
            "motion"

            and

            event.value
            >
            0.72
        ),

        lambda event:
            "Possible obstacle detected"
    )


    # --------------------------------------------------------
    # RULE 2
    # --------------------------------------------------------

    memory.add_rule(

        "VisualChangeRule",

        lambda event: (

            event.source
            ==
            "vision"

            and

            event.context
            ==
            "navigation"

            and

            event.novelty
            >
            0.43
        ),

        lambda event:
            "Unexpected visual change"
    )


    # --------------------------------------------------------
    # RULE 3
    # --------------------------------------------------------

    memory.add_rule(

        "AudioAlertRule",

        lambda event: (

            event.source
            ==
            "audio"

            and

            event.value
            >
            0.88
        ),

        lambda event:
            "Strong audio event"
    )


    # --------------------------------------------------------
    # RULE 4
    # --------------------------------------------------------

    memory.add_rule(

        "TemperatureAlertRule",

        lambda event: (

            event.source
            ==
            "temperature"

            and

            event.value
            >
            0.96
        ),

        lambda event:
            "Extreme temperature event"
    )


    return memory


# ============================================================
# 11. SIMULATED ENVIRONMENT
# ============================================================

def generate_events(
    number_of_events: int = 120,
    seed: int = 7
) -> List[Event]:


    random.seed(
        seed
    )


    sources = [

        "vision",

        "audio",

        "motion",

        "temperature"
    ]


    contexts = [

        "navigation",

        "conversation",

        "idle"
    ]


    previous_values = {

        source: 0.10

        for source in sources
    }


    events = []


    # --------------------------------------------------------
    # GENERATE EVENT STREAM
    # --------------------------------------------------------

    for event_id in range(
        number_of_events
    ):


        source = random.choice(
            sources
        )


        context = random.choice(
            contexts
        )


        value = random.random()


        previous = (
            previous_values[
                source
            ]
        )


        previous_values[
            source
        ] = value


        novelty = abs(

            value
            -
            previous
        )


        # ====================================================
        # HIDDEN EXPERIMENTAL GROUND TRUTH
        #
        # IMPORTANT:
        #
        # The filtering systems do NOT directly use this.
        #
        # It exists only so that we can evaluate them later.
        # ====================================================


        important = (

            (

                source
                ==
                "motion"

                and

                value
                >
                0.72
            )


            or


            (

                source
                ==
                "vision"

                and

                context
                ==
                "navigation"

                and

                novelty
                >
                0.43
            )


            or


            (

                source
                ==
                "audio"

                and

                context
                ==
                "conversation"

                and

                value
                >
                0.88
            )


            or


            (

                source
                ==
                "temperature"

                and

                value
                >
                0.96
            )
        )


        event = Event(

            event_id=
                event_id,

            source=
                source,

            value=
                value,

            previous_value=
                previous,

            context=
                context,

            important=
                important
        )


        events.append(
            event
        )


    return events


# ============================================================
# 12. EVALUATION
# ============================================================

def evaluate(
    selected: List[Event],
    all_events: List[Event]
) -> Dict[str, float]:


    # --------------------------------------------------------
    # ACTUALLY IMPORTANT EVENTS
    # --------------------------------------------------------

    important_ids = {

        event.event_id

        for event in all_events

        if event.important
    }


    # --------------------------------------------------------
    # EVENTS SELECTED BY SYSTEM
    # --------------------------------------------------------

    selected_ids = {

        event.event_id

        for event in selected
    }


    # --------------------------------------------------------
    # TRUE POSITIVES
    # --------------------------------------------------------

    true_positive = len(

        important_ids

        &

        selected_ids
    )


    # --------------------------------------------------------
    # FALSE POSITIVES
    # --------------------------------------------------------

    false_positive = len(

        selected_ids

        -

        important_ids
    )


    # --------------------------------------------------------
    # FALSE NEGATIVES
    # --------------------------------------------------------

    false_negative = len(

        important_ids

        -

        selected_ids
    )


    # --------------------------------------------------------
    # RECALL
    # --------------------------------------------------------

    if (
        true_positive
        +
        false_negative
        >
        0
    ):


        recall = (

            true_positive

            /

            (
                true_positive
                +
                false_negative
            )
        )


    else:

        recall = 0.0


    # --------------------------------------------------------
    # PRECISION
    # --------------------------------------------------------

    if (
        true_positive
        +
        false_positive
        >
        0
    ):


        precision = (

            true_positive

            /

            (
                true_positive
                +
                false_positive
            )
        )


    else:

        precision = 0.0


    # --------------------------------------------------------
    # FALSE-NEGATIVE RATE
    # --------------------------------------------------------

    if (
        true_positive
        +
        false_negative
        >
        0
    ):


        false_negative_rate = (

            false_negative

            /

            (
                true_positive
                +
                false_negative
            )
        )


    else:

        false_negative_rate = 0.0


    return {

        "processed":
            len(selected_ids),

        "important_detected":
            true_positive,

        "false_positive":
            false_positive,

        "false_negative":
            false_negative,

        "recall":
            recall,

        "precision":
            precision,

        "false_negative_rate":
            false_negative_rate
    }


# ============================================================
# 13. STATISTICS HELPERS
# ============================================================

def mean(
    values: List[float]
) -> float:


    if values:

        return statistics.mean(
            values
        )


    return 0.0



def std(
    values: List[float]
) -> float:


    if len(values) > 1:

        return statistics.stdev(
            values
        )


    return 0.0



def compare_scores(
    challenger: float,
    baseline: float
) -> str:


    if challenger > baseline:

        return "win"


    elif challenger < baseline:

        return "loss"


    else:

        return "tie"


# ============================================================
# 14. FULL ABLATION BENCHMARK
# ============================================================

def benchmark(
    trials: int = 200,
    budget: int = 20
) -> None:


    # --------------------------------------------------------
    # STORE ALL RESULTS
    # --------------------------------------------------------

    systems = {

        "Fixed Filter":
            [],

        "HI Context Only":
            [],

        "HI Same-Source Recheck":
            [],

        "HI Cross-Modal Recheck":
            []
    }


    # --------------------------------------------------------
    # HEAD-TO-HEAD COUNTERS
    # --------------------------------------------------------

    wins = {


        "HI Context Only": {

            "win": 0,

            "tie": 0,

            "loss": 0
        },


        "HI Same-Source Recheck": {

            "win": 0,

            "tie": 0,

            "loss": 0
        },


        "HI Cross-Modal Recheck": {

            "win": 0,

            "tie": 0,

            "loss": 0
        }
    }


    cross_vs_context = {

        "win": 0,

        "tie": 0,

        "loss": 0
    }


    # ========================================================
    # RUN EXPERIMENTS
    # ========================================================

    for seed in range(
        trials
    ):


        # ----------------------------------------------------
        # SAME ENVIRONMENT FOR ALL SYSTEMS
        # ----------------------------------------------------

        events = generate_events(
            seed=seed
        )


        # ----------------------------------------------------
        # SYSTEM A
        #
        # FIXED FILTER
        # ----------------------------------------------------

        fixed = FixedFilter(

            processing_budget=
                budget
        )


        # ----------------------------------------------------
        # SYSTEM B
        #
        # CONTEXT ONLY
        # ----------------------------------------------------

        context_only = HumanoidFilter(

            processing_budget=
                budget,

            mode=
                "context_only"
        )


        # ----------------------------------------------------
        # SYSTEM C
        #
        # SAME-SOURCE RECHECKING
        # ----------------------------------------------------

        same_source = HumanoidFilter(

            processing_budget=
                budget,

            recheck_fraction=
                0.30,

            mode=
                "same_source"
        )


        # ----------------------------------------------------
        # SYSTEM D
        #
        # CROSS-MODAL RECHECKING
        # ----------------------------------------------------

        cross_modal = HumanoidFilter(

            processing_budget=
                budget,

            recheck_fraction=
                0.30,

            mode=
                "cross_modal"
        )


        # ----------------------------------------------------
        # SELECT EVENTS
        # ----------------------------------------------------

        selections = {


            "Fixed Filter":

                fixed.select(
                    events
                ),


            "HI Context Only":

                context_only.select(
                    events
                ),


            "HI Same-Source Recheck":

                same_source.select(
                    events
                ),


            "HI Cross-Modal Recheck":

                cross_modal.select(
                    events
                )
        }


        # ----------------------------------------------------
        # EVALUATE ALL SYSTEMS
        # ----------------------------------------------------

        results = {


            name:

                evaluate(
                    selected,
                    events
                )


            for (
                name,
                selected
            )

            in selections.items()
        }


        # ----------------------------------------------------
        # STORE RESULTS
        # ----------------------------------------------------

        for (
            name,
            result
        ) in results.items():


            systems[
                name
            ].append(
                result
            )


        # ----------------------------------------------------
        # FIXED RECALL
        # ----------------------------------------------------

        fixed_recall = (

            results[
                "Fixed Filter"
            ][
                "recall"
            ]
        )


        # ----------------------------------------------------
        # COMPARE EACH HI VERSION AGAINST FIXED
        # ----------------------------------------------------

        for name in wins:


            comparison = compare_scores(

                results[
                    name
                ][
                    "recall"
                ],

                fixed_recall
            )


            wins[
                name
            ][
                comparison
            ] += 1


        # ----------------------------------------------------
        # CROSS-MODAL VS CONTEXT ONLY
        # ----------------------------------------------------

        cross_comparison = compare_scores(


            results[
                "HI Cross-Modal Recheck"
            ][
                "recall"
            ],


            results[
                "HI Context Only"
            ][
                "recall"
            ]
        )


        cross_vs_context[
            cross_comparison
        ] += 1


    # ========================================================
    # PRINT MAIN RESULTS
    # ========================================================

    print()

    print(
        "=" * 60
    )

    print(
        "HUMANOID INTELLIGENCE - ENHANCED ABLATION EXPERIMENT"
    )

    print(
        "=" * 60
    )


    print(
        "Trials:",
        trials
    )


    print(
        "Processing budget per system:",
        budget
    )


    # --------------------------------------------------------
    # SYSTEM RESULTS
    # --------------------------------------------------------

    for (
        name,
        result_list
    ) in systems.items():


        recalls = [

            result["recall"]

            for result
            in result_list
        ]


        precisions = [

            result["precision"]

            for result
            in result_list
        ]


        false_negatives = [

            result[
                "false_negative"
            ]

            for result
            in result_list
        ]


        false_positives = [

            result[
                "false_positive"
            ]

            for result
            in result_list
        ]


        fn_rates = [

            result[
                "false_negative_rate"
            ]

            for result
            in result_list
        ]


        print()

        print(
            "-" * 60
        )

        print(
            name
        )

        print(
            "-" * 60
        )


        print(

            "Average recall:",

            round(
                mean(recalls),
                3
            )
        )


        print(

            "Recall standard deviation:",

            round(
                std(recalls),
                3
            )
        )


        print(

            "Average precision:",

            round(
                mean(precisions),
                3
            )
        )


        print(

            "Precision standard deviation:",

            round(
                std(precisions),
                3
            )
        )


        print(

            "Average false negatives:",

            round(
                mean(
                    false_negatives
                ),
                2
            )
        )


        print(

            "Average false positives:",

            round(
                mean(
                    false_positives
                ),
                2
            )
        )


        print(

            "Average false-negative rate:",

            round(
                mean(
                    fn_rates
                ),
                3
            )
        )


    # ========================================================
    # HEAD-TO-HEAD VS FIXED
    # ========================================================

    print()

    print(
        "=" * 60
    )

    print(
        "HEAD-TO-HEAD RECALL VS FIXED FILTER"
    )

    print(
        "=" * 60
    )


    for (
        name,
        record
    ) in wins.items():


        print()

        print(
            name
        )


        print(
            "Wins:",
            record["win"]
        )


        print(
            "Ties:",
            record["tie"]
        )


        print(
            "Losses:",
            record["loss"]
        )


    # ========================================================
    # CROSS-MODAL VS CONTEXT ONLY
    # ========================================================

    print()

    print(
        "=" * 60
    )

    print(
        "CROSS-MODAL RECHECK VS CONTEXT-ONLY"
    )

    print(
        "=" * 60
    )


    print(
        "Wins:",
        cross_vs_context["win"]
    )


    print(
        "Ties:",
        cross_vs_context["tie"]
    )


    print(
        "Losses:",
        cross_vs_context["loss"]
    )


    # ========================================================
    # AVERAGE RECALL IMPROVEMENT
    # ========================================================

    fixed_mean_recall = mean(

        [

            result["recall"]

            for result

            in systems[
                "Fixed Filter"
            ]
        ]
    )


    context_mean_recall = mean(

        [

            result["recall"]

            for result

            in systems[
                "HI Context Only"
            ]
        ]
    )


    same_mean_recall = mean(

        [

            result["recall"]

            for result

            in systems[
                "HI Same-Source Recheck"
            ]
        ]
    )


    cross_mean_recall = mean(

        [

            result["recall"]

            for result

            in systems[
                "HI Cross-Modal Recheck"
            ]
        ]
    )


    print()

    print(
        "=" * 60
    )

    print(
        "AVERAGE RECALL IMPROVEMENT"
    )

    print(
        "=" * 60
    )


    print(

        "Context Only vs Fixed:",

        round(

            context_mean_recall

            -

            fixed_mean_recall,

            3
        )
    )


    print(

        "Same-Source Recheck vs Fixed:",

        round(

            same_mean_recall

            -

            fixed_mean_recall,

            3
        )
    )


    print(

        "Cross-Modal Recheck vs Fixed:",

        round(

            cross_mean_recall

            -

            fixed_mean_recall,

            3
        )
    )


    print(

        "Cross-Modal Recheck vs Context Only:",

        round(

            cross_mean_recall

            -

            context_mean_recall,

            3
        )
    )


# ============================================================
# 15. PROCEDURAL MEMORY DEMONSTRATION
#
# This shows what happens AFTER HI decides
# which events deserve processing.
# ============================================================

def procedural_memory_demo(
    seed: int = 7,
    budget: int = 20
) -> None:


    events = generate_events(
        seed=seed
    )


    hi = HumanoidFilter(

        processing_budget=
            budget,

        recheck_fraction=
            0.30,

        mode=
            "cross_modal"
    )


    memory = (
        build_memory()
    )


    selected = (
        hi.select(
            events
        )
    )


    print()

    print(
        "=" * 60
    )

    print(
        "PROCEDURAL MEMORY DEMO"
    )

    print(
        "=" * 60
    )


    action_count = 0


    for event in selected:


        responses = (
            memory.execute(
                event
            )
        )


        for response in responses:


            action_count += 1


            print(

                f"Event "
                f"{event.event_id:03d}"

                f" | "

                f"{event.source:11s}"

                f" | value="
                f"{event.value:.3f}"

                f" | context="
                f"{event.context:12s}"

                f" | "

                f"{response['rule']}"

                f" -> "

                f"{response['action']}"
            )


    if action_count == 0:

        print(
            "No procedural rules fired in this trial."
        )


# ============================================================
# 16. MAIN
# ============================================================

if __name__ == "__main__":


    # --------------------------------------------------------
    # MAIN SCIENTIFIC EXPERIMENT
    # --------------------------------------------------------

    benchmark(

        trials=
            200,

        budget=
            20
    )


    # --------------------------------------------------------
    # PROCEDURAL-MEMORY EXAMPLE
    # --------------------------------------------------------

    procedural_memory_demo(

        seed=
            7,

        budget=
            20
    )


HUMANOID INTELLIGENCE - ENHANCED ABLATION EXPERIMENT
Trials: 200
Processing budget per system: 20

------------------------------------------------------------
Fixed Filter
------------------------------------------------------------
Average recall: 0.548
Recall standard deviation: 0.134
Average precision: 0.373
Precision standard deviation: 0.107
Average false negatives: 6.45
Average false positives: 12.54
Average false-negative rate: 0.452

------------------------------------------------------------
HI Context Only
------------------------------------------------------------
Average recall: 0.601
Recall standard deviation: 0.125
Average precision: 0.409
Precision standard deviation: 0.109
Average false negatives: 5.73
Average false positives: 11.82
Average false-negative rate: 0.399

------------------------------------------------------------
HI Same-Source Recheck
------------------------------------------------------------
Average recall: 0.589
Recall standard deviation: 0.127
A

In [8]:
# ============================================================
# HUMANOID INTELLIGENCE - HI-0.2
#
# CAUSAL / MULTI-SENSOR EVENT SIMULATION
#
# No ML
# No DL
# No neural network
#
# We compare:
#
# 1. Fixed Filter
# 2. Context-Aware HI
# 3. Cross-Modal HI
#
# ============================================================


from dataclasses import dataclass
from typing import List, Optional
import random
import statistics


# ============================================================
# 1. EVENT
# ============================================================

@dataclass
class Event:

    event_id: int

    # Time when event happened
    time: float

    # vision / audio / motion / temperature
    source: str

    # Current sensor strength
    value: float

    # Previous reading from same sensor
    previous_value: float

    # Current environmental context
    context: str

    # Hidden answer key.
    # The HI system must NOT directly use this.
    important: bool = False

    # Hidden incident number.
    # Only used for evaluation.
    incident_id: Optional[int] = None


    # --------------------------------------------------------
    # How much did the signal change?
    # --------------------------------------------------------

    @property
    def novelty(self):

        difference = abs(
            self.value
            -
            self.previous_value
        )

        return min(
            difference,
            1.0
        )


    # --------------------------------------------------------
    # How strong is the signal?
    # --------------------------------------------------------

    @property
    def magnitude(self):

        return min(
            abs(self.value),
            1.0
        )


# ============================================================
# 2. CREATE A MORE REALISTIC ENVIRONMENT
# ============================================================

def generate_environment(
    seed=7,
    duration=60.0,
    noise_events=180,
    incidents=12
):


    rng = random.Random(seed)


    # --------------------------------------------------------
    # The system is performing one main task.
    # --------------------------------------------------------

    active_context = rng.choice(
        [
            "navigation",
            "conversation"
        ]
    )


    sources = [

        "vision",
        "audio",
        "motion",
        "temperature"

    ]


    contexts = [

        "navigation",
        "conversation",
        "idle"

    ]


    raw_events = []


    # ========================================================
    # 3. BACKGROUND NOISE
    #
    # These events do NOT represent important incidents.
    # ========================================================

    for _ in range(noise_events):


        event_time = rng.uniform(
            0,
            duration
        )


        source = rng.choice(
            sources
        )


        context = rng.choice(
            contexts
        )


        # ----------------------------------------------------
        # Sometimes noise can be very strong.
        #
        # This is important because a simple filter may
        # incorrectly think:
        #
        # "Strong signal = important."
        # ----------------------------------------------------

        if rng.random() < 0.12:

            value = rng.uniform(
                0.75,
                1.00
            )

        else:

            value = rng.uniform(
                0.05,
                0.65
            )


        raw_events.append(

            [

                event_time,
                source,
                value,
                context,

                False,      # not important

                None        # no incident
            ]
        )


    # ========================================================
    # 4. CREATE REAL INCIDENTS
    #
    # Each incident generates MULTIPLE related signals.
    # ========================================================


    incident_times = [

        rng.uniform(
            2,
            duration - 2
        )

        for _ in range(
            incidents
        )
    ]


    incident_times.sort()


    for incident_id, center_time in enumerate(
        incident_times
    ):


        # ====================================================
        # NAVIGATION INCIDENT
        #
        # Example:
        #
        # obstacle appears
        #
        # vision
        # motion
        # audio
        #
        # occur very close in time.
        # ====================================================

        if active_context == "navigation":


            components = [


                (
                    "vision",

                    rng.uniform(
                        0.48,
                        0.72
                    ),

                    rng.uniform(
                        -0.12,
                        0.00
                    )
                ),


                (
                    "motion",

                    rng.uniform(
                        0.52,
                        0.78
                    ),

                    rng.uniform(
                        0.00,
                        0.10
                    )
                ),


                (
                    "audio",

                    rng.uniform(
                        0.40,
                        0.68
                    ),

                    rng.uniform(
                        0.08,
                        0.22
                    )
                )

            ]


        # ====================================================
        # CONVERSATION INCIDENT
        #
        # Example:
        #
        # important person speaks / gestures
        #
        # audio
        # vision
        # motion
        #
        # appear together.
        # ====================================================

        else:


            components = [


                (
                    "audio",

                    rng.uniform(
                        0.50,
                        0.78
                    ),

                    rng.uniform(
                        -0.10,
                        0.00
                    )
                ),


                (
                    "vision",

                    rng.uniform(
                        0.42,
                        0.68
                    ),

                    rng.uniform(
                        0.02,
                        0.14
                    )
                ),


                (
                    "motion",

                    rng.uniform(
                        0.38,
                        0.64
                    ),

                    rng.uniform(
                        0.08,
                        0.22
                    )
                )

            ]


        # ----------------------------------------------------
        # Add all signals produced by this incident.
        # ----------------------------------------------------

        for (
            source,
            value,
            time_offset
        ) in components:


            raw_events.append(

                [

                    center_time
                    +
                    time_offset,

                    source,

                    value,

                    active_context,

                    True,

                    incident_id
                ]

            )


    # ========================================================
    # 5. SORT EVENTS BY TIME
    # ========================================================

    raw_events.sort(
        key=lambda event:
            event[0]
    )


    # ========================================================
    # 6. BUILD FINAL EVENT OBJECTS
    # ========================================================

    previous_values = {

        "vision": 0.10,

        "audio": 0.10,

        "motion": 0.10,

        "temperature": 0.10

    }


    events = []


    for event_id, raw in enumerate(
        raw_events
    ):


        event_time = raw[0]

        source = raw[1]

        value = raw[2]

        context = raw[3]

        important = raw[4]

        incident_id = raw[5]


        previous = (
            previous_values[source]
        )


        previous_values[source] = (
            value
        )


        event = Event(

            event_id=
                event_id,

            time=
                event_time,

            source=
                source,

            value=
                value,

            previous_value=
                previous,

            context=
                context,

            important=
                important,

            incident_id=
                incident_id

        )


        events.append(
            event
        )


    return (
        events,
        active_context
    )


# ============================================================
# 7. FIXED FILTER
# ============================================================

class FixedFilter:


    def __init__(
        self,
        processing_budget=30
    ):

        self.processing_budget = (
            processing_budget
        )


    def score(
        self,
        event
    ):


        return (

            0.50
            *
            event.novelty

            +

            0.50
            *
            event.magnitude

        )


    def select(
        self,
        events
    ):


        ranked = sorted(

            events,

            key=self.score,

            reverse=True

        )


        return ranked[
            :self.processing_budget
        ]


# ============================================================
# 8. CONTEXT-AWARE HI
# ============================================================

class ContextHI:


    def __init__(
        self,
        processing_budget=30,
        active_context="navigation"
    ):


        self.processing_budget = (
            processing_budget
        )


        self.active_context = (
            active_context
        )


        self.source_priority = {

            "vision": 0.90,

            "audio": 0.70,

            "motion": 1.00,

            "temperature": 0.50

        }


    # --------------------------------------------------------
    # First attention score
    # --------------------------------------------------------

    def score(
        self,
        event
    ):


        priority = (

            self.source_priority.get(

                event.source,

                0.50

            )

        )


        if (
            event.context
            ==
            self.active_context
        ):

            context_relevance = 1.0

        else:

            context_relevance = 0.25


        score = (

            0.40
            *
            event.novelty

            +

            0.30
            *
            event.magnitude

            +

            0.20
            *
            priority

            +

            0.10
            *
            context_relevance

        )


        return min(
            score,
            1.0
        )


    def select(
        self,
        events
    ):


        ranked = sorted(

            events,

            key=self.score,

            reverse=True

        )


        return ranked[
            :self.processing_budget
        ]


# ============================================================
# 9. CROSS-MODAL HI
# ============================================================

class CrossModalHI(
    ContextHI
):


    def __init__(
        self,
        processing_budget=30,
        active_context="navigation",
        recheck_fraction=0.30,
        time_window=0.35
    ):


        super().__init__(

            processing_budget,
            active_context

        )


        self.recheck_fraction = (
            recheck_fraction
        )


        # Events within 0.35 seconds
        # are considered temporally related.

        self.time_window = (
            time_window
        )


        # ----------------------------------------------------
        # Relationships between sensor types.
        #
        # These are prototype values.
        # ----------------------------------------------------

        self.cross_modal_links = {


            "vision": {

                "motion": 1.00,

                "audio": 0.70

            },


            "motion": {

                "vision": 1.00,

                "audio": 0.90

            },


            "audio": {

                "motion": 0.90,

                "vision": 0.70

            },


            "temperature": {

                "vision": 0.40

            }

        }


    # ========================================================
    # 10. RECHECK
    # ========================================================

    def recheck_score(
        self,
        event,
        events
    ):


        original_score = (
            self.score(
                event
            )
        )


        best_evidence = 0.0


        supporting_modalities = set()


        # ----------------------------------------------------
        # Search nearby events.
        # ----------------------------------------------------

        for other in events:


            # Don't compare event with itself.

            if (
                other.event_id
                ==
                event.event_id
            ):

                continue


            time_difference = abs(

                other.time
                -
                event.time

            )


            # Too far away in time?

            if (
                time_difference
                >
                self.time_window
            ):

                continue


            # ------------------------------------------------
            # Is there a known relationship?
            # ------------------------------------------------

            relationship = (

                self.cross_modal_links

                .get(
                    event.source,
                    {}
                )

                .get(
                    other.source,
                    0.0
                )

            )


            if relationship <= 0:

                continue


            # ------------------------------------------------
            # Closer events provide stronger evidence.
            # ------------------------------------------------

            temporal_strength = (

                1.0

                -

                (
                    time_difference
                    /
                    self.time_window
                )

            )


            evidence = (

                relationship

                *

                temporal_strength

                *

                (

                    0.55
                    *
                    other.novelty

                    +

                    0.45
                    *
                    other.magnitude

                )

            )


            if (
                evidence
                >
                best_evidence
            ):

                best_evidence = (
                    evidence
                )


            # ------------------------------------------------
            # Remember which different sensors support us.
            # ------------------------------------------------

            if evidence > 0.15:

                supporting_modalities.add(

                    other.source

                )


        # ----------------------------------------------------
        # Multiple different sensors = stronger evidence.
        # ----------------------------------------------------

        diversity_boost = min(

            0.03
            *
            len(
                supporting_modalities
            ),

            0.09

        )


        # ----------------------------------------------------
        # Final reconsideration score
        # ----------------------------------------------------

        final_score = (

            original_score

            +

            0.30
            *
            best_evidence

            +

            diversity_boost

        )


        return min(
            final_score,
            1.0
        )


    # ========================================================
    # 11. SELECT EVENTS
    # ========================================================

    def select(
        self,
        events
    ):


        # ----------------------------------------------------
        # First-pass ranking
        # ----------------------------------------------------

        ranked = sorted(

            events,

            key=self.score,

            reverse=True

        )


        # ----------------------------------------------------
        # Example:
        #
        # budget = 30
        #
        # 70% first pass
        # 30% rechecking
        # ----------------------------------------------------

        first_budget = round(

            self.processing_budget

            *

            (
                1.0
                -
                self.recheck_fraction
            )

        )


        recheck_budget = (

            self.processing_budget

            -

            first_budget

        )


        # ----------------------------------------------------
        # Immediately accept strongest events.
        # ----------------------------------------------------

        selected = ranked[
            :first_budget
        ]


        # ----------------------------------------------------
        # Take lower-ranked candidates.
        # ----------------------------------------------------

        candidate_pool = ranked[

            first_budget

            :

            first_budget
            +
            (
                self.processing_budget
                *
                4
            )

        ]


        # ----------------------------------------------------
        # Re-evaluate using nearby different sensors.
        # ----------------------------------------------------

        reconsidered = sorted(

            candidate_pool,

            key=lambda event:

                self.recheck_score(
                    event,
                    events
                ),

            reverse=True

        )


        # ----------------------------------------------------
        # Fill remaining processing slots.
        # ----------------------------------------------------

        selected.extend(

            reconsidered[
                :recheck_budget
            ]

        )


        return selected[
            :self.processing_budget
        ]


# ============================================================
# 12. EVALUATION
# ============================================================

def evaluate(
    selected,
    all_events
):


    important_ids = {

        event.event_id

        for event in all_events

        if event.important

    }


    selected_ids = {

        event.event_id

        for event in selected

    }


    true_positive = len(

        important_ids
        &
        selected_ids

    )


    false_positive = len(

        selected_ids
        -
        important_ids

    )


    false_negative = len(

        important_ids
        -
        selected_ids

    )


    # --------------------------------------------------------
    # EVENT RECALL
    # --------------------------------------------------------

    if (
        true_positive
        +
        false_negative
    ) > 0:


        recall = (

            true_positive

            /

            (
                true_positive
                +
                false_negative
            )

        )


    else:

        recall = 0.0


    # --------------------------------------------------------
    # PRECISION
    # --------------------------------------------------------

    if (
        true_positive
        +
        false_positive
    ) > 0:


        precision = (

            true_positive

            /

            (
                true_positive
                +
                false_positive
            )

        )


    else:

        precision = 0.0


    # ========================================================
    # INCIDENT DETECTION
    #
    # Did the system notice at least ONE signal
    # belonging to the real incident?
    # ========================================================

    incident_ids = {

        event.incident_id

        for event in all_events

        if (
            event.incident_id
            is not None
        )

    }


    detected_incidents = {

        event.incident_id

        for event in selected

        if (
            event.incident_id
            is not None
        )

    }


    incident_recall = (

        len(
            detected_incidents
        )

        /

        len(
            incident_ids
        )

    )


    # ========================================================
    # MULTI-SIGNAL INCIDENT DETECTION
    #
    # Did we notice at least TWO signals
    # from the same incident?
    # ========================================================

    incident_counts = {

        incident_id: 0

        for incident_id
        in incident_ids

    }


    for event in selected:


        if (
            event.incident_id
            is not None
        ):


            incident_counts[
                event.incident_id
            ] += 1


    multi_signal_detected = sum(

        1

        for count
        in incident_counts.values()

        if count >= 2

    )


    multi_signal_recall = (

        multi_signal_detected

        /

        len(
            incident_ids
        )

    )


    return {

        "recall":
            recall,

        "precision":
            precision,

        "incident_recall":
            incident_recall,

        "multi_signal_recall":
            multi_signal_recall,

        "true_positive":
            true_positive,

        "false_positive":
            false_positive,

        "false_negative":
            false_negative

    }


# ============================================================
# 13. BENCHMARK
# ============================================================

def benchmark(
    trials=300,
    processing_budget=30
):


    results = {

        "Fixed":
            [],

        "Context HI":
            [],

        "Cross-Modal HI":
            []

    }


    cross_wins = 0

    cross_ties = 0

    cross_losses = 0


    # ========================================================
    # RUN MANY DIFFERENT ENVIRONMENTS
    # ========================================================

    for seed in range(
        trials
    ):


        events, active_context = (

            generate_environment(
                seed=seed
            )

        )


        fixed = FixedFilter(

            processing_budget=
                processing_budget

        )


        context_hi = ContextHI(

            processing_budget=
                processing_budget,

            active_context=
                active_context

        )


        cross_hi = CrossModalHI(

            processing_budget=
                processing_budget,

            active_context=
                active_context,

            recheck_fraction=
                0.30,

            time_window=
                0.35

        )


        fixed_result = evaluate(

            fixed.select(
                events
            ),

            events

        )


        context_result = evaluate(

            context_hi.select(
                events
            ),

            events

        )


        cross_result = evaluate(

            cross_hi.select(
                events
            ),

            events

        )


        results[
            "Fixed"
        ].append(
            fixed_result
        )


        results[
            "Context HI"
        ].append(
            context_result
        )


        results[
            "Cross-Modal HI"
        ].append(
            cross_result
        )


        # ----------------------------------------------------
        # Compare cross-modal vs context-only
        # ----------------------------------------------------

        if (
            cross_result["recall"]
            >
            context_result["recall"]
        ):

            cross_wins += 1


        elif (
            cross_result["recall"]
            ==
            context_result["recall"]
        ):

            cross_ties += 1


        else:

            cross_losses += 1


    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print()

    print(
        "=" * 65
    )

    print(
        "HUMANOID INTELLIGENCE - HI-0.2"
    )

    print(
        "CAUSAL MULTI-SENSOR SIMULATION"
    )

    print(
        "=" * 65
    )


    print(
        "Trials:",
        trials
    )


    print(
        "Processing budget:",
        processing_budget
    )


    for (
        system_name,
        system_results
    ) in results.items():


        recalls = [

            result["recall"]

            for result
            in system_results

        ]


        precisions = [

            result["precision"]

            for result
            in system_results

        ]


        incident_recalls = [

            result["incident_recall"]

            for result
            in system_results

        ]


        multi_recalls = [

            result["multi_signal_recall"]

            for result
            in system_results

        ]


        print()

        print(
            "-" * 65
        )

        print(
            system_name
        )

        print(
            "-" * 65
        )


        print(

            "Average event recall:",

            round(
                statistics.mean(
                    recalls
                ),
                3
            )

        )


        print(

            "Average precision:",

            round(
                statistics.mean(
                    precisions
                ),
                3
            )

        )


        print(

            "Incident recall:",

            round(
                statistics.mean(
                    incident_recalls
                ),
                3
            )

        )


        print(

            "Multi-signal incident recall:",

            round(
                statistics.mean(
                    multi_recalls
                ),
                3
            )

        )


    print()

    print(
        "=" * 65
    )

    print(
        "CROSS-MODAL VS CONTEXT-ONLY"
    )

    print(
        "=" * 65
    )


    print(
        "Cross-modal wins:",
        cross_wins
    )


    print(
        "Ties:",
        cross_ties
    )


    print(
        "Cross-modal losses:",
        cross_losses
    )


# ============================================================
# 14. SHOW ONE REAL INCIDENT
#
# This lets us SEE what the generated environment looks like.
# ============================================================

def show_incident_example(
    seed=7
):


    events, active_context = (

        generate_environment(
            seed=seed
        )

    )


    print()

    print(
        "=" * 65
    )

    print(
        "EXAMPLE OF ONE SIMULATED INCIDENT"
    )

    print(
        "=" * 65
    )


    print(
        "Active context:",
        active_context
    )


    example_incident = 0


    for event in events:


        if (
            event.incident_id
            ==
            example_incident
        ):


            print(

                "Incident:",

                event.incident_id,

                "| time:",

                round(
                    event.time,
                    3
                ),

                "| source:",

                event.source,

                "| value:",

                round(
                    event.value,
                    3
                ),

                "| novelty:",

                round(
                    event.novelty,
                    3
                )

            )


# ============================================================
# 15. RUN EVERYTHING
# ============================================================

if __name__ == "__main__":


    benchmark(

        trials=300,

        processing_budget=30

    )


    show_incident_example(
        seed=7
    )


HUMANOID INTELLIGENCE - HI-0.2
CAUSAL MULTI-SENSOR SIMULATION
Trials: 300
Processing budget: 30

-----------------------------------------------------------------
Fixed
-----------------------------------------------------------------
Average event recall: 0.16
Average precision: 0.192
Incident recall: 0.406
Multi-signal incident recall: 0.07

-----------------------------------------------------------------
Context HI
-----------------------------------------------------------------
Average event recall: 0.263
Average precision: 0.315
Incident recall: 0.591
Multi-signal incident recall: 0.177

-----------------------------------------------------------------
Cross-Modal HI
-----------------------------------------------------------------
Average event recall: 0.34
Average precision: 0.408
Incident recall: 0.64
Multi-signal incident recall: 0.313

CROSS-MODAL VS CONTEXT-ONLY
Cross-modal wins: 286
Ties: 12
Cross-modal losses: 2

EXAMPLE OF ONE SIMULATED INCIDENT
Active context: convers